# backend/api.ipynb (scaffold)

API notebook: outline FastAPI endpoints (`/search`, `/stats`, `/ws/metrics`, `/pause`, `/resume`) and WebSocket push every 1 second.

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any, Literal

from fastapi import FastAPI, Query, WebSocket, WebSocketDisconnect
from pydantic import BaseModel, Field

from .crawler import Crawler, CrawlerConfig
from .index import IndexEntry
from .metrics import CrawlMetrics
from .search import search_index


def _metrics_payload(metrics: CrawlMetrics) -> dict[str, Any]:
    return {
        "crawled": metrics.crawled,
        "queued": metrics.queued,
        "dropped": metrics.dropped,
        "workers_active": metrics.workers_active,
        "workers_max": metrics.workers_max,
        "back_pressure": metrics.back_pressure,
        "status": metrics.status,
        "elapsed_seconds": metrics.elapsed_seconds(),
    }


class CrawlStartRequest(BaseModel):
    seed_url: str = Field(..., min_length=1)
    k: int = Field(..., ge=0, description="Max recursive depth")
    scope: Literal["same-domain", "same-origin", "all"] = "same-domain"
    workers: int = Field(10, ge=1, le=200)
    queue_size: int = Field(1000, ge=1, le=200000)
    timeout_s: int = Field(10, ge=1, le=120)


def create_app() -> FastAPI:
    app = FastAPI(title="Google-in-a-Day (FastAPI)")

    # Shared in-memory state (event-loop only).
    app.state.visited = set[str]()
    app.state.index = dict[str, IndexEntry]()
    app.state.metrics = CrawlMetrics(workers_max=10)
    app.state.crawl_task: asyncio.Task[tuple[dict[str, IndexEntry], CrawlMetrics]] | None = None
    app.state.crawl_lock = asyncio.Lock()

    async def _stop_crawl_if_running() -> None:
        task = app.state.crawl_task
        if task is None:
            return
        task.cancel()
        await asyncio.gather(task, return_exceptions=True)
        app.state.crawl_task = None

    @app.get("/search")
    async def search_endpoint(q: str = Query(..., min_length=1)) -> dict[str, Any]:
        results = await search_index(app.state.index, q)
        return {
            "query": q,
            "results": results,
            "total": len(results),
            "index_size": len(app.state.index),
        }

    @app.post("/crawl/start")
    async def crawl_start(req: CrawlStartRequest) -> dict[str, Any]:
        async with app.state.crawl_lock:
            await _stop_crawl_if_running()

            # Reset shared state for a clean restart.
            app.state.visited = set[str]()
            app.state.index = dict[str, IndexEntry]()
            app.state.metrics = CrawlMetrics(workers_max=req.workers)

            config = CrawlerConfig(
                seed_url=req.seed_url,
                max_depth_k=req.k,
                scope=req.scope,
                max_workers=req.workers,
                queue_maxsize=req.queue_size,
                timeout_s=req.timeout_s,
            )
            crawler = Crawler(config)
            app.state.crawl_task = asyncio.create_task(
                crawler.run(
                    index=app.state.index,
                    visited=app.state.visited,
                    metrics=app.state.metrics,
                )
            )

            return {"ok": True, "metrics": _metrics_payload(app.state.metrics)}

    @app.post("/crawl/stop")
    async def crawl_stop() -> dict[str, Any]:
        async with app.state.crawl_lock:
            await _stop_crawl_if_running()
            return {
                "ok": True,
                "metrics": _metrics_payload(app.state.metrics),
            }

    @app.websocket("/ws/metrics")
    async def ws_metrics(ws: WebSocket) -> None:
        await ws.accept()
        try:
            while True:
                metrics = app.state.metrics
                await ws.send_json(_metrics_payload(metrics))
                await asyncio.sleep(1)
        except WebSocketDisconnect:
            return

    @app.get("/stats")
    async def stats_endpoint() -> dict[str, Any]:
        return _metrics_payload(app.state.metrics)

    return app
